# M35 — Physics-Informed Acoustic Loss Constraints
**Model ID:** M35  
**Novelty Extension:** §4.10 — Physics-Informed Acoustic Constraints  
**Contributor:** Barshon  
**Project:** OWMTL

## Objective
Incorporate domain-specific acoustic physics constraints into the loss function:
- **Wheeze:** sustained tonal energy in 100–1000 Hz, duration > 100ms
- **Crackle:** short transient energy < 20ms, broadband
A soft penalty term encourages the model to produce predictions consistent with known spectral-temporal signatures of respiratory sounds.

## Section 1: Environment Setup & Dependencies

In [1]:
# ============================================================
# Section 1: Environment Setup & Dependencies
# ============================================================
import os, sys, re, time, json, math, glob, random, shutil, io, zipfile, tempfile
import base64, datetime
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, precision_recall_fscore_support,
    classification_report
)

# Seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print(f'Device: {DEVICE} ({GPU_NAME})')
print(f'PyTorch: {torch.__version__} | Python: {sys.version.split()[0]}')

Device: cuda (Tesla T4)
PyTorch: 2.10.0+cu128 | Python: 3.12.13


## Section 2: Configuration & Path Resolution

In [2]:
# ============================================================
# Section 2: Configuration & Path Resolution
# ============================================================
# ---- Auto-detect Platform ----
if os.path.exists('/kaggle'):
    PLATFORM = 'Kaggle'
    BASE_DIR = '/kaggle/working'
elif os.path.exists('/content'):
    PLATFORM = 'Colab'
    BASE_DIR = '/content'
else:
    PLATFORM = 'Local'
    BASE_DIR = '.'
print(f'Platform: {PLATFORM}')

# ---- Google Drive Mount (Colab) ----
DRIVE_DIR = None
if PLATFORM == 'Colab':
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        DRIVE_DIR = '/content/drive/MyDrive/OWMTL/M35'
        os.makedirs(DRIVE_DIR, exist_ok=True)
    except Exception as e:
        print(f'Drive mount skipped ({e})')

# ---- ICBHI Dataset Path Resolution ----
POSSIBLE_ROOTS = [
    '/kaggle/input/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files',
    '/kaggle/input/respiratory-sound-database/audio_and_txt_files',
    '/kaggle/input/respiratory-sound-database/Respiratory_Sound_Database/audio_and_txt_files',
    '/kaggle/input/icbhi-2017-respiratory-sound-database/audio_and_txt_files',
    '/content/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files',
    '/content/drive/MyDrive/respiratory-sound-database/audio_and_txt_files',
    '/content/drive/MyDrive/OWMTL/data/audio_and_txt_files',
    './data/audio_and_txt_files',
]
DATA_ROOT = next((p for p in POSSIBLE_ROOTS if os.path.exists(p)), None)
if DATA_ROOT is None and os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        if any(f.endswith('.wav') for f in files) and any(f.endswith('.txt') for f in files):
            DATA_ROOT = root
            print(f'Dynamic Kaggle resolution: {DATA_ROOT}')
            break
if DATA_ROOT and os.path.exists(DATA_ROOT):
    print(f'✅ ICBHI dataset verified: {DATA_ROOT}')
else:
    print(f'⚠️ DATA_ROOT not found — set DATA_ROOT manually')

# ---- M12/M2 Backbone Checkpoint Resolution ----
def resolve_checkpoint(candidates):
    return next((p for p in candidates if p and os.path.exists(p)), None)
M2_CKPT_PATH = resolve_checkpoint([
    '/kaggle/input/datasets/barshonbasak/m2-checkpoint/best_model.pth',
    '/content/M2_best_model.pth',
    '/kaggle/input/m2-checkpoint/best_model.pth',
    '/kaggle/input/owmtl-m2/best_model.pth',
    '/kaggle/input/m2-best-model/best_model.pth',
    '/content/drive/MyDrive/OWMTL/M2/best_model.pth',
    '../M2/best_model.pth',
    os.path.join(BASE_DIR, 'best_model.pth'),
])

CFG = {
    'model_id': 'M35',
    'model_name': 'Physics-Informed Acoustic Loss (Optimized)',
    'contributor': 'Barshon',
    'seed': SEED,
    # Shared Audio Parameters (Protocol §2)
    'sample_rate': 16000,
    'duration_s': 8.0,
    'n_mels': 128,
    'n_fft': 1024,
    'hop_length': 160,
    'win_length': 400,
    'f_min': 50,
    'f_max': 2000,
    'n_samples': int(16000 * 8.0),
    'n_frames': 1 + math.floor(128000 / 160),
    # Sound Event Classes (4)
    'sound_classes': ['Normal', 'Crackle', 'Wheeze', 'Both'],
    'num_classes': 4,
    # Training Hyperparameters (Optimized for Fine-Tuning)
    'batch_size': 32,
    'num_epochs': 30,
    'lr': 0.0001,  # Reduced from 0.001 to preserve pretrained features
    'weight_decay': 0.0001,
    'dropout': 0.4,
    'physics_penalty_weight': 0.05,
    'warmup_epochs': 10,
    'freeze_early_blocks': True,
    'architecture': 'M2_PhysicsLoss_v2',
    'data_root': DATA_ROOT,
    'm2_ckpt_path': M2_CKPT_PATH,
    'ckpt_dir': os.path.join(BASE_DIR, 'checkpoints_M35'),
    'results_dir': os.path.join(BASE_DIR, 'results_M35'),
}

os.makedirs(CFG['ckpt_dir'], exist_ok=True)
os.makedirs(CFG['results_dir'], exist_ok=True)

print(f"\n{'='*60}")
print(f'M35 CONFIGURATION — Physics-Informed Acoustic Loss (Optimized)')
print(f"{'='*60}")
for k, v in CFG.items():
    if 'path' in k or 'dir' in k:
        print(f'  {k}: {v}')
print(f"{'='*60}")

Platform: Kaggle
Dynamic Kaggle resolution: /kaggle/input/datasets/vbookshelf/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files
✅ ICBHI dataset verified: /kaggle/input/datasets/vbookshelf/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files

M35 CONFIGURATION — Physics-Informed Acoustic Loss (Optimized)
  m2_ckpt_path: /kaggle/input/datasets/barshonbasak/m2-checkpoint/best_model.pth
  ckpt_dir: /kaggle/working/checkpoints_M35
  results_dir: /kaggle/working/results_M35


## Section 3: Real ICBHI Audio Loading & Patient-Independent Splitting

In [3]:
# ============================================================
# Section 3: Real ICBHI Audio Loading & Patient-Independent Splitting
# ============================================================
try:
    import librosa
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'librosa'])
    import librosa

def extract_log_mel(wav_path, start, end, cfg):
    """Extract normalized log-mel spectrogram from a respiratory cycle."""
    sr, n_samples = cfg['sample_rate'], cfg['n_samples']
    try:
        audio, _ = librosa.load(wav_path, sr=sr, offset=start,
                                duration=max(end - start, 0.05), mono=True)
    except Exception:
        return np.zeros((1, cfg['n_mels'], cfg['n_frames']), dtype=np.float32)
    if len(audio) == 0:
        return np.zeros((1, cfg['n_mels'], cfg['n_frames']), dtype=np.float32)
    
    # Pad or trim to fixed length
    if len(audio) < n_samples:
        audio = np.tile(audio, math.ceil(n_samples / len(audio)))[:n_samples]
    else:
        audio = audio[:n_samples]
        
    mel = librosa.feature.melspectrogram(
        y=audio, sr=sr, n_mels=cfg['n_mels'], n_fft=cfg['n_fft'],
        hop_length=cfg['hop_length'], win_length=cfg['win_length'],
        fmin=cfg['f_min'], fmax=cfg['f_max'], power=2.0)
    log_mel = librosa.power_to_db(mel, ref=np.max)
    log_mel = (log_mel - log_mel.min()) / (log_mel.max() - log_mel.min() + 1e-8)
    
    T = log_mel.shape[1]
    if T < cfg['n_frames']:
        log_mel = np.pad(log_mel, ((0, 0), (0, cfg['n_frames'] - T)), mode='constant')
    else:
        log_mel = log_mel[:, :cfg['n_frames']]
    return log_mel[np.newaxis, :, :].astype(np.float32)

def parse_annotation_file(txt_path):
    """Parse ICBHI annotation file into cycle list with labels."""
    cycles = []
    with open(txt_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 4: continue
            try:
                start, end = float(parts[0]), float(parts[1])
                crackle, wheeze = int(parts[2]), int(parts[3])
            except ValueError: continue
            if end <= start: continue
            
            if crackle == 0 and wheeze == 0: label = 0
            elif crackle == 1 and wheeze == 0: label = 1
            elif crackle == 0 and wheeze == 1: label = 2
            else: label = 3
            cycles.append({'start': start, 'end': end, 'label': label})
    return cycles

def build_icbhi_splits(data_root, cfg):
    """Load all ICBHI cycles and perform official patient-independent split."""
    wav_paths = sorted(glob.glob(os.path.join(data_root, '*.wav')))
    if not wav_paths:
        raise FileNotFoundError(f'No .wav files under {data_root}')
    
    rows = []
    for wav_path in wav_paths:
        stem = os.path.splitext(os.path.basename(wav_path))[0]
        txt_path = os.path.join(data_root, stem + '.txt')
        if not os.path.exists(txt_path): continue
        try: pid = int(stem.split('_')[0])
        except (ValueError, IndexError): continue
        
        cycles = parse_annotation_file(txt_path)
        for c in cycles:
            rows.append({
                'wav_path': wav_path, 'stem': stem, 'patient_id': pid,
                'start': c['start'], 'end': c['end'], 'sound_label': c['label']
            })
            
    df = pd.DataFrame(rows)
    all_pids = sorted(df['patient_id'].unique())
    np.random.seed(SEED)
    np.random.shuffle(all_pids)
    
    n_train = int(len(all_pids) * 0.70)
    train_pids = set(all_pids[:n_train])
    test_pids = set(all_pids[n_train:])
    
    df_train = df[df['patient_id'].isin(train_pids)].reset_index(drop=True)
    df_test = df[df['patient_id'].isin(test_pids)].reset_index(drop=True)
    return df_train, df_test

class RealICBHI_SoundDataset(Dataset):
    """ICBHI respiratory sound event dataset loading real .wav audio."""
    def __init__(self, df, cfg):
        self.df = df.reset_index(drop=True)
        self.cfg = cfg
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        spec = extract_log_mel(row['wav_path'], row['start'], row['end'], self.cfg)
        return torch.from_numpy(spec), torch.tensor(row['sound_label'], dtype=torch.long)

df_train, df_test = build_icbhi_splits(CFG['data_root'], CFG)
print(f'Train set: {len(df_train)} cycles across {df_train["patient_id"].nunique()} patients')
print(f'Test set:  {len(df_test)} cycles across {df_test["patient_id"].nunique()} patients')

train_ds = RealICBHI_SoundDataset(df_train, CFG)
test_ds = RealICBHI_SoundDataset(df_test, CFG)
train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True, drop_last=True)
test_loader = DataLoader(test_ds, batch_size=CFG['batch_size'], shuffle=False)

# Class weights (inverse frequency)
class_counts = df_train['sound_label'].value_counts().sort_index().values
class_weights = 1.0 / (class_counts.astype(np.float32) + 1e-6)
class_weights = class_weights / class_weights.sum()
CLASS_WEIGHTS = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
print(f'Class counts:  {class_counts}')
print(f'Class weights: {class_weights.round(4)}')

Train set: 4149 cycles across 88 patients
Test set:  2749 cycles across 38 patients
Class counts:  [2363  977  489  320]
Class weights: [0.064  0.1547 0.3091 0.4723]


## Section 4: Physics-Informed Loss & Model Architecture

In [4]:
# ---- M2 CNN Backbone (same as M30/M2 architecture) ----
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, pool=(2, 2)):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=pool),
        )
    def forward(self, x): return self.block(x)

class M2_CNN(nn.Module):
    """M2 CNN Backbone — 5-block architecture with 768-dim embedding."""
    def __init__(self, num_classes=4, depth=5, base_width=48, dropout=0.4, fc_dim=128):
        super().__init__()
        channels = [base_width * (2 ** i) for i in range(depth)]  # [48, 96, 192, 384, 768]
        blocks, in_ch = [], 1
        for out_ch in channels:
            blocks.append(ConvBlock(in_ch, out_ch))
            in_ch = out_ch
        self.encoder = nn.Sequential(*blocks)
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.Linear(channels[-1], fc_dim),
            nn.ReLU(inplace=True),
            nn.Linear(fc_dim, num_classes),
        )
        self.embedding_dim = channels[-1]  # 768
    def get_embedding(self, x):
        return self.gap(self.encoder(x)).flatten(1)
    def forward(self, x):
        emb = self.get_embedding(x)
        emb = self.dropout(emb)
        return self.head(emb)

def smart_load_checkpoint(path, device):
    """Load checkpoint handling both .pth and .zip formats."""
    if not os.path.exists(path):
        raise FileNotFoundError(f'File not found: {path}')
    if zipfile.is_zipfile(path):
        try:
            with zipfile.ZipFile(path, 'r') as z:
                names = z.namelist()
                target = 'best_model.pth'
                if target not in names:
                    target = next((n for n in names if n.endswith('.pth')), None)
                if target:
                    with z.open(target) as f:
                        return torch.load(io.BytesIO(f.read()), map_location=device, weights_only=False)
        except Exception:
            pass
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except Exception:
        return torch.load(path, map_location=device, weights_only=True)

# ---- Physically-Grounded Acoustic Loss (PG-Loss) ----

class VectorizedAcousticPhysicsLoss(nn.Module):
    """
    Physically-Grounded Acoustic Loss for Respiratory Sounds (v2).
    
    1. Wheeze Tonal Energy: Wiener Spectral Flatness (SF) along frequency per time frame.
       Pure tones have SF -> 0, broadband noise has SF -> 1.
       Wheezes: 100-1000 Hz band (mel bins ~10 to ~80).
    
    2. Crackle Transient Energy: Temporal PAPR & Max Temporal Gradient.
       Transients (<20ms) produce high temporal spikes PAPR >> 1 and large frame differences.
    """
    def __init__(self, n_mels=128, f_min=50, f_max=2000, penalty_weight=0.05,
                 wheeze_sf_thresh=0.4, crackle_papr_thresh=2.5):
        super().__init__()
        self.penalty_weight = penalty_weight
        self.wheeze_low_bin = 10
        self.wheeze_high_bin = 80
        self.wheeze_sf_thresh = wheeze_sf_thresh
        self.crackle_papr_thresh = crackle_papr_thresh

    def forward(self, spectrograms, predictions, labels):
        """
        spectrograms: [B, 1, n_mels, T] log-mel spectrograms
        predictions: [B, 4] logits
        labels: [B] ground truth sound labels
        """
        B = spectrograms.shape[0]
        if B == 0:
            return torch.zeros(1, device=spectrograms.device)

        specs = spectrograms.squeeze(1)  # [B, n_mels, T]
        power_specs = torch.exp(specs * 3.0)  # un-log scale approximation for energy ratios

        # --- 1. Wheeze Feature: Wiener Spectral Flatness in 100-1000 Hz band ---
        wheeze_band = power_specs[:, self.wheeze_low_bin:self.wheeze_high_bin, :] + 1e-7
        log_mean = torch.mean(torch.log(wheeze_band), dim=1)
        arith_mean = torch.mean(wheeze_band, dim=1)
        geom_mean = torch.exp(log_mean)
        spectral_flatness = (geom_mean / (arith_mean + 1e-7)).mean(dim=1)

        # --- 2. Crackle Feature: Temporal PAPR ---
        max_temporal = power_specs.max(dim=2).values.mean(dim=1)
        avg_temporal = power_specs.mean(dim=(1, 2)) + 1e-7
        temporal_papr = max_temporal / avg_temporal

        # --- 3. Bounded Sigmoid Penalty Calculation ---
        probs = F.softmax(predictions, dim=1)
        wheeze_prob = probs[:, 2] + probs[:, 3]
        crackle_prob = probs[:, 1] + probs[:, 3]

        wheeze_mask = ((labels == 2) | (labels == 3)).float()
        crackle_mask = ((labels == 1) | (labels == 3)).float()

        # Wheeze penalty: if SF > wheeze_sf_thresh (not tonal enough when wheeze is predicted/labeled)
        wheeze_penalty = wheeze_prob * torch.sigmoid(10.0 * (spectral_flatness - self.wheeze_sf_thresh)) * wheeze_mask

        # Crackle penalty: if PAPR < crackle_papr_thresh (not transient enough when crackle is predicted/labeled)
        crackle_penalty = crackle_prob * torch.sigmoid(5.0 * (self.crackle_papr_thresh - temporal_papr)) * crackle_mask

        total_penalty = (wheeze_penalty + crackle_penalty).mean()
        return self.penalty_weight * total_penalty

# ---- M35 Model Setup ----
model = M2_CNN(num_classes=CFG['num_classes'], dropout=CFG['dropout']).to(DEVICE)
physics_loss_fn = VectorizedAcousticPhysicsLoss(penalty_weight=CFG['physics_penalty_weight']).to(DEVICE)

if CFG['m2_ckpt_path']:
    try:
        ckpt_m2 = smart_load_checkpoint(CFG['m2_ckpt_path'], DEVICE)
        sd = ckpt_m2.get('model_state', ckpt_m2.get('model_state_dict', ckpt_m2))
        model.load_state_dict(sd, strict=False)
        print(f'✅ Loaded M2 weights')
    except Exception as e:
        print(f'⚠️ M2 load: {e}')

if CFG.get('freeze_early_blocks', True):
    for name, param in model.named_parameters():
        if any(block_name in name for block_name in ['encoder.0', 'encoder.1', 'encoder.2']):
            param.requires_grad = False
    print('🔒 Frozen Conv Blocks 1-3 to preserve low-level acoustic feature representations')

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total Params:     {total_params:,}')
print(f'Trainable Params: {trainable_params:,}')

✅ Loaded M2 weights
🔒 Frozen Conv Blocks 1-3 to preserve low-level acoustic feature representations
Total Params:     3,627,476
Trainable Params: 3,419,012


## Section 5: Training Loop with Physics-Informed Loss

In [5]:
def eval_epoch(model, loader, criterion, device):
    """Evaluate model on one epoch — returns loss, acc, f1, icbhi_score, targets, preds."""
    model.eval()
    total_loss, all_preds, all_targets = 0.0, [], []
    with torch.no_grad():
        for specs, labels in loader:
            specs, labels = specs.to(device), labels.to(device)
            outputs = model(specs)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * len(labels)
            preds = outputs.argmax(dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(labels.cpu().numpy())
    avg_loss = total_loss / max(len(loader.dataset), 1)
    acc = accuracy_score(all_targets, all_preds)
    macro_f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)
    cm = confusion_matrix(all_targets, all_preds, labels=list(range(4)))
    sens = np.diag(cm) / (cm.sum(axis=1) + 1e-6)
    macro_sens = np.mean(sens)
    specs_list = []
    for i in range(4):
        tp = cm[i, i]; fp = cm[:, i].sum() - tp; fn = cm[i, :].sum() - tp
        tn = cm.sum() - tp - fp - fn
        specs_list.append(tn / (tn + fp + 1e-6))
    macro_spec = np.mean(specs_list)
    icbhi_score = (macro_sens + macro_spec) / 2.0
    return avg_loss, acc, macro_f1, icbhi_score, all_targets, all_preds

def get_physics_weight(epoch, total_warmup=10, max_weight=CFG['physics_penalty_weight']):
    """Cosine Loss Warmup schedule for physics penalty weight."""
    if epoch <= total_warmup:
        return max_weight * 0.5 * (1.0 - math.cos(math.pi * epoch / total_warmup))
    return max_weight

# ============================================================
# Section 5: Training Loop with Physics-Informed Loss (Optimized)
# ============================================================

criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS)
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['num_epochs'])

history = []
best_score = 0.0
start_epoch = 1
best_ckpt_path = os.path.join(CFG['ckpt_dir'], 'best_model.pth')
last_ckpt_path = os.path.join(CFG['ckpt_dir'], 'last_checkpoint.pth')

# Auto-Resume (§6 & §11)
resume_path = last_ckpt_path if os.path.exists(last_ckpt_path) else None
if resume_path and os.path.exists(resume_path):
    try:
        ckpt_res = torch.load(resume_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt_res['model_state'])
        optimizer.load_state_dict(ckpt_res['optimizer_state'])
        scheduler.load_state_dict(ckpt_res['scheduler_state'])
        start_epoch = int(ckpt_res['epoch']) + 1
        best_score = float(ckpt_res.get('best_score', 0.0))
        history = ckpt_res.get('history', [])
        print(f'✅ Resumed from Epoch {start_epoch-1}')
    except Exception as e:
        print(f'Resume failed: {e}')

print(f'\n--- PHYSICS-INFORMED TRAINING (OPTIMIZED): EPOCH {start_epoch} TO {CFG["num_epochs"]} ---')
start_time = time.time()

for epoch in range(start_epoch, CFG['num_epochs'] + 1):
    model.train()
    train_loss, train_physics = 0.0, 0.0
    correct_train = 0
    total_train = 0
    all_train_preds, all_train_targets = [], []
    t0 = time.time()
    
    current_phys_weight = get_physics_weight(epoch, total_warmup=CFG['warmup_epochs'], max_weight=CFG['physics_penalty_weight'])
    physics_loss_fn.penalty_weight = current_phys_weight
    
    for specs, labels in train_loader:
        specs, labels = specs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(specs)
        ce_loss = criterion(outputs, labels)
        phys_penalty = physics_loss_fn(specs, outputs, labels)
        total_loss = ce_loss + phys_penalty
        total_loss.backward()
        optimizer.step()
        
        train_loss += ce_loss.item() * len(labels)
        train_physics += phys_penalty.item() * len(labels)
        preds = outputs.argmax(dim=-1)
        correct_train += (preds == labels).sum().item()
        total_train += len(labels)
        all_train_preds.extend(preds.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
    scheduler.step()
    n = max(len(train_loader.dataset), 1)
    train_loss /= n
    train_physics /= n
    train_acc = correct_train / max(total_train, 1)
    train_f1 = f1_score(all_train_targets, all_train_preds, average='macro', zero_division=0)
    epoch_time = time.time() - t0

    val_loss, val_acc, val_f1, val_icbhi, _, _ = eval_epoch(model, test_loader, criterion, DEVICE)

    history.append({
        'epoch': int(epoch),
        'train_loss': float(train_loss),
        'train_physics_penalty': float(train_physics),
        'val_loss': float(val_loss),
        'train_accuracy': float(train_acc),
        'val_accuracy': float(val_acc),
        'train_f1_macro': float(train_f1),
        'val_f1_macro': float(val_f1),
        'val_icbhi_score': float(val_icbhi),
        'lr': float(optimizer.param_groups[0]['lr']),
        'epoch_time_s': float(epoch_time),
    })

    # Save checkpoints
    torch.save({
        'epoch': int(epoch), 'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(), 'scheduler_state': scheduler.state_dict(),
        'best_score': float(best_score), 'history': history,
    }, last_ckpt_path)

    is_best = val_icbhi > best_score
    if is_best:
        best_score = val_icbhi
        torch.save({
            'epoch': int(epoch), 'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(), 'icbhi_score': float(val_icbhi),
        }, best_ckpt_path)

    if epoch % 5 == 0 or epoch == 1 or is_best:
        star = ' 🏆 BEST' if is_best else ''
        print(f'Epoch {epoch:02d}/{CFG["num_epochs"]} | CE: {train_loss:.4f} Physics(w={current_phys_weight:.4f}): {train_physics:.6f} | '
              f'VLoss: {val_loss:.4f} | VICBHI: {val_icbhi:.4f}{star}')

total_train_time = time.time() - start_time
print(f'\n✅ Physics-informed training complete in {total_train_time:.1f}s. Best ICBHI: {best_score:.4f}')


--- PHYSICS-INFORMED TRAINING (OPTIMIZED): EPOCH 1 TO 30 ---
Epoch 01/30 | CE: 0.7974 Physics(w=0.0012): 0.000333 | VLoss: 0.7008 | VICBHI: 0.7918 🏆 BEST
Epoch 05/30 | CE: 0.6895 Physics(w=0.0250): 0.007118 | VLoss: 0.7228 | VICBHI: 0.7839
Epoch 10/30 | CE: 0.6234 Physics(w=0.0500): 0.014467 | VLoss: 0.8119 | VICBHI: 0.7510
Epoch 15/30 | CE: 0.5797 Physics(w=0.0500): 0.014737 | VLoss: 0.7498 | VICBHI: 0.7636
Epoch 20/30 | CE: 0.5430 Physics(w=0.0500): 0.015152 | VLoss: 0.7552 | VICBHI: 0.7621
Epoch 25/30 | CE: 0.5189 Physics(w=0.0500): 0.015150 | VLoss: 0.7712 | VICBHI: 0.7556
Epoch 30/30 | CE: 0.5137 Physics(w=0.0500): 0.015150 | VLoss: 0.7717 | VICBHI: 0.7579

✅ Physics-informed training complete in 4212.7s. Best ICBHI: 0.7918


## Section 6: Comprehensive Evaluation & Visualizations

In [6]:
# ============================================================
# Section 6: Comprehensive Evaluation & Visualizations
# ============================================================
# Load best checkpoint
ckpt = torch.load(best_ckpt_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['model_state'])
best_ep = int(ckpt['epoch'])

loss, acc, f1_val, icbhi, targets, preds = eval_epoch(model, test_loader, criterion, DEVICE)

cm = confusion_matrix(targets, preds, labels=list(range(4)))
cm_norm = cm.astype(np.float32) / (cm.sum(axis=1, keepdims=True) + 1e-6)

prec_macro = precision_score(targets, preds, average='macro', zero_division=0)
rec_macro = recall_score(targets, preds, average='macro', zero_division=0)

spec_per_class = []
for i in range(4):
    tp = cm[i, i]; fp = cm[:, i].sum() - tp; fn = cm[i, :].sum() - tp
    tn = cm.sum() - tp - fp - fn
    spec_per_class.append(float(tn / (tn + fp + 1e-6)))
spec_macro = float(np.mean(spec_per_class))

prec_per = precision_score(targets, preds, average=None, zero_division=0, labels=list(range(4)))
rec_per = recall_score(targets, preds, average=None, zero_division=0, labels=list(range(4)))
f1_per = f1_score(targets, preds, average=None, zero_division=0, labels=list(range(4)))
support_per = [int(np.sum(np.array(targets) == i)) for i in range(4)]

# Model size
with io.BytesIO() as b:
    torch.save(model.state_dict(), b)
    model_size_mb = len(b.getvalue()) / (1024 * 1024)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

# Inference time
model.eval()
dummy = torch.randn(1, 1, CFG['n_mels'], CFG['n_frames']).to(DEVICE)
times_inf = []
with torch.no_grad():
    for _ in range(50):
        t0 = time.time()
        _ = model(dummy)
        times_inf.append((time.time() - t0) * 1000)
inf_ms = float(np.median(times_inf))

print(f'\n{"="*60}')
print(f'M35 FINAL EVALUATION RESULTS')
print(f'{"="*60}')
print(f'  Best Epoch:       {best_ep}')
print(f'  Test Accuracy:    {acc:.4f}')
print(f'  Macro Precision:  {prec_macro:.4f}')
print(f'  Macro Recall:     {rec_macro:.4f}')
print(f'  Macro F1:         {f1_val:.4f}')
print(f'  Macro Specificity:{spec_macro:.4f}')
print(f'  ICBHI Score:      {icbhi:.4f}')
print(f'  Model Size:       {model_size_mb:.2f} MB')
print(f'  Total Params:     {total_params:,}')
print(f'  Trainable Params: {trainable_params:,}')
print(f'  Inference Time:   {inf_ms:.2f} ms/sample')
print(f'{"="*60}')

# ---- Visualization (§5 Required Plots) ----
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

epochs_list = [h['epoch'] for h in history]
train_l = [h['train_loss'] for h in history]
val_l = [h['val_loss'] for h in history]
val_a = [h['val_accuracy'] for h in history]
val_f1_list = [h['val_f1_macro'] for h in history]

# Plot 1: Loss curves
ax = axes[0, 0]
ax.plot(epochs_list, train_l, 'b-o', markersize=3, label='Train Loss')
ax.plot(epochs_list, val_l, 'r-s', markersize=3, label='Val Loss')
ax.axvline(best_ep, color='green', linestyle='--', alpha=0.7, label=f'Best Epoch ({best_ep})')
ax.set_title('M35 — Loss Curves')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.legend(); ax.grid(True, alpha=0.3)

# Plot 2: Accuracy & ICBHI Score
ax = axes[0, 1]
val_icbhi_list = [h['val_icbhi_score'] for h in history]
ax.plot(epochs_list, val_a, 'g-o', markersize=3, label='Val Accuracy')
ax.plot(epochs_list, val_icbhi_list, 'm-s', markersize=3, label='Val ICBHI Score')
ax.axvline(best_ep, color='green', linestyle='--', alpha=0.7)
ax.set_title('M35 — Performance Curves')
ax.set_xlabel('Epoch'); ax.set_ylabel('Score')
ax.legend(); ax.grid(True, alpha=0.3)

# Plot 3: Raw Confusion Matrix
ax = axes[1, 0]
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CFG['sound_classes'], yticklabels=CFG['sound_classes'], ax=ax)
ax.set_title('Raw Confusion Matrix')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')

# Plot 4: Normalized Confusion Matrix
ax = axes[1, 1]
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Greens',
            xticklabels=CFG['sound_classes'], yticklabels=CFG['sound_classes'], ax=ax)
ax.set_title('Normalized Confusion Matrix')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')

plt.suptitle('M35 — M35 Evaluation', fontsize=14, y=1.01)
plt.tight_layout()

for d in sorted(set([CFG['results_dir'], BASE_DIR])):
    fig.savefig(os.path.join(d, 'm35_results.png'), dpi=150, bbox_inches='tight')
print('Saved: m35_results.png')
plt.show()
plt.close()


M35 FINAL EVALUATION RESULTS
  Best Epoch:       1
  Test Accuracy:    0.6766
  Macro Precision:  0.6405
  Macro Recall:     0.6982
  Macro F1:         0.6491
  Macro Specificity:0.8854
  ICBHI Score:      0.7918
  Model Size:       13.86 MB
  Total Params:     3,627,476
  Trainable Params: 3,419,012
  Inference Time:   0.79 ms/sample
Saved: m35_results.png


## Section 7: Exporting Protocol-Compliant Results JSON

In [7]:
# ============================================================
# Section 7: Exporting Protocol-Compliant Results JSON (§4 Schema)
# ============================================================
per_class_dict = {}
for i, cls_name in enumerate(CFG['sound_classes']):
    per_class_dict[cls_name] = {
        'precision': round(float(prec_per[i]), 4),
        'recall': round(float(rec_per[i]), 4),
        'f1': round(float(f1_per[i]), 4),
        'specificity': round(spec_per_class[i], 4),
        'support': support_per[i],
    }

results = {
    'meta': {
        'model_id': 'M35',
        'model_name': 'Physics-Informed Acoustic Loss',
        'contributor': 'Barshon',
        'date_completed': datetime.datetime.now().strftime('%Y-%m-%d'),
        'is_augmented': False,
        'augmentation_method': 'none',
        'notes': 'Novelty Search §4.10 - Physics-Informed Acoustic Constraints',
    },
    'config': {
        'sample_rate': CFG['sample_rate'],
        'n_mels': CFG['n_mels'],
        'batch_size': CFG['batch_size'],
        'num_epochs': CFG['num_epochs'],
        'lr': CFG['lr'],
        'optimizer': 'Adam',
        'scheduler': 'CosineAnnealingLR',
        'architecture': CFG['architecture'],
        'seed': CFG['seed'],
    },
    'environment': {
        'platform': PLATFORM,
        'gpu_name': GPU_NAME,
        'pytorch_version': torch.__version__,
        'python_version': sys.version.split()[0],
    },
    'dataset_info': {
        'dataset': 'ICBHI_2017',
        'data_source': 'real_audio',
        'train_samples': int(len(df_train)),
        'test_samples': int(len(df_test)),
        'train_patients': int(df_train['patient_id'].nunique()),
        'test_patients': int(df_test['patient_id'].nunique()),
        'split_method': 'patient_independent_70_30',
    },
    'efficiency': {
        'total_params': int(total_params),
        'trainable_params': int(trainable_params),
        'model_size_mb': round(float(model_size_mb), 2),
        'training_time_total_s': round(float(total_train_time), 2),
        'training_time_per_epoch_s_avg': round(float(total_train_time / max(CFG['num_epochs'], 1)), 2),
        'gpu_name': GPU_NAME,
        'inference_time_ms_per_sample': round(inf_ms, 2),
    },
    'best_epoch': {
        'epoch': int(best_ep),
        'primary_metric': 'icbhi_score',
        'primary_metric_value': round(float(icbhi), 4),
    },
    'best_metrics': {
        'accuracy': round(float(acc), 4),
        'precision_macro': round(float(prec_macro), 4),
        'recall_macro': round(float(rec_macro), 4),
        'f1_macro': round(float(f1_val), 4),
        'specificity_macro': round(spec_macro, 4),
        'icbhi_score': round(float(icbhi), 4),
        'per_class': per_class_dict,
        'confusion_matrix_raw': cm.tolist(),
        'confusion_matrix_normalized': cm_norm.round(4).tolist(),
    },
    'ablation': {
        'ablation_group': 'physics_informed_constraints',
        'ablation_role': 'variant',
        'baseline_model_id': 'M2',
        'variable_changed': 'loss: CE + acoustic physics penalty (wheeze tonal + crackle transient)',
        'variables_held_constant': [
            'loss_function: inverse_frequency_CrossEntropyLoss',
            'data_split: patient_independent_70_30',
            'seed: 42',
            'preprocessing: 128mel_16kHz_8s',
        ],
        'component_flags': {
            'has_sound_event_head': True,
            'has_disease_head': False,
            'has_cross_task_consistency': False,
            'has_cqkd_regularization': False,
            'has_openmax_rejection': False,
            'owl_stage': 0,
            'compression_clusters': None,
            'has_physics_informed_loss': True,
        },
        'loss_weights': {
            'sound_event_weight': 1.0,
            'disease_weight': None,
            'consistency_weight': None,
        },
    },
    'training_history': history,
}

json_path = os.path.join(CFG['results_dir'], 'results_M35.json')
with open(json_path, 'w') as f:
    json.dump(results, f, indent=2, default=str)
print(f'\u2705 Saved: {json_path}')

# Also save to model folder if running locally
local_dir = os.path.join(BASE_DIR, "Barshon's", "M35")
if os.path.isdir(local_dir):
    local_json = os.path.join(local_dir, 'results_M35.json')
    with open(local_json, 'w') as f:
        json.dump(results, f, indent=2, default=str)
    print(f'\u2705 Saved copy: {local_json}')

✅ Saved: /kaggle/working/results_M35/results_M35.json


## Section 8: Summary & Key Takeaways
**Model:** M35 — Physics-Informed Acoustic Loss
**Novelty Item:** §4.10 — Physics-Informed Acoustic Constraints

**Key Results:**
- All metrics computed on **real ICBHI audio** with patient-independent splits
- Protocol-compliant `results_M35.json` with all 9 required blocks
- Best model checkpoint saved at `checkpoints_M35/best_model.pth`

**What This Means for the Novelty Claim:**
Tests whether encoding known acoustic physics (wheeze tonal signatures, crackle transient patterns) as loss penalties improves or regularizes classification.

## Section 8: Team Handoff & Downloads

In [8]:
# ============================================================
# Section 8: Team Handoff & One-Click File Downloads (§11.D)
# ============================================================
from IPython.display import display, FileLink
print("=" * 60)
print("OFFICIAL PROTOCOL OUTPUTS READY FOR DOWNLOAD")
print("=" * 60)

protocol_files = sorted(
    glob.glob(os.path.join(CFG['ckpt_dir'], 'best_model.pth')) +
    glob.glob(os.path.join(CFG['results_dir'], 'results_M35.json')) +
    glob.glob(os.path.join(CFG['results_dir'], '*.png')))

for fpath in protocol_files:
    if os.path.exists(fpath):
        size_mb = round(os.path.getsize(fpath) / (1024 * 1024), 2)
        print(f"Ready: {os.path.basename(fpath):<25} ({size_mb} MB)")
        display(FileLink(fpath))
    else:
        print(f"Missing: {os.path.basename(fpath)}")

bundle_dir = os.path.join(BASE_DIR, 'protocol_bundle_M35')
if protocol_files:
    os.makedirs(bundle_dir, exist_ok=True)
    for fpath in protocol_files:
        if os.path.exists(fpath):
            shutil.copy2(fpath, os.path.join(bundle_dir, os.path.basename(fpath)))
    zip_path = shutil.make_archive(
        os.path.join(BASE_DIR, 'm35_handoff_bundle'), 'zip', bundle_dir)
    size_zip = round(os.path.getsize(zip_path) / (1024 * 1024), 2)
    print(f"\nZIP bundle ({size_zip} MB):")
    display(FileLink('m35_handoff_bundle.zip'))
print("=" * 60)

OFFICIAL PROTOCOL OUTPUTS READY FOR DOWNLOAD
Ready: best_model.pth            (39.95 MB)


/kaggle/working/checkpoints_M35/best_model.pth

Ready: m35_results.png           (0.23 MB)


/kaggle/working/results_M35/m35_results.png

Ready: results_M35.json          (0.02 MB)


/kaggle/working/results_M35/results_M35.json


ZIP bundle (37.04 MB):


/kaggle/working/m35_handoff_bundle.zip